# ICU LOS Classification Demo

This lightweight demo loads a synthetic sample dataset, loads the saved three-class ICU LOS classifier, and generates predictions. It does not require restricted MIMIC-IV data.

In [ ]:
from pathlib import Path
import sys

import joblib
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, f1_score

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

sample_path = REPO_ROOT / 'data' / 'sample' / 'icu_los_classification_sample.csv'
model_path = REPO_ROOT / 'models' / 'icu_los_classifier_sample.joblib'

df = pd.read_csv(sample_path)
artifact = joblib.load(model_path)
model = artifact['model']
feature_columns = artifact['feature_columns']
target_column = artifact['target_column']
target_labels = artifact['target_labels']

X = df[feature_columns]
y = df[target_column]
pred = model.predict(X)
proba = model.predict_proba(X)

predictions = df[['subject_id', 'stay_id', target_column]].copy()
predictions['predicted_los_category'] = pred
for idx, label in enumerate(model.classes_):
    predictions[f'prob_class_{label}'] = proba[:, idx]

print('Macro F1:', round(f1_score(y, pred, average='macro', zero_division=0), 3))
print('Weighted F1:', round(f1_score(y, pred, average='weighted', zero_division=0), 3))
print('Balanced accuracy:', round(balanced_accuracy_score(y, pred), 3))
print('\nConfusion matrix:')
print(confusion_matrix(y, pred, labels=[0, 1, 2]))
print('\nClass labels:', target_labels)
predictions.head(10)